# Thesis: Entity-Aware A-RAG with Evidence Verification

**Đề tài**: Nghiên cứu cải tiến mô hình A-RAG dựa trên theo dõi thực thể và kiểm chứng bằng chứng trong hỏi đáp đa bước

**Branch**: `thesis-entity-evidence-arag`

**Thứ tự chạy**: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11

## Cell 1: Clone Repo / Pull Code

In [ ]:
import os

REPO_URL = "https://github.com/trangdx2602/arag.git"
REPO_DIR = "/content/arag"
BRANCH = "thesis-entity-evidence-arag"

if os.path.exists(REPO_DIR):
    print("Repo already exists — pulling latest...")
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone {REPO_URL} {REPO_DIR}
    !cd {REPO_DIR} && git checkout {BRANCH}

!ls {REPO_DIR}

# Change working directory so relative paths in YAML configs resolve correctly
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Cell 2: Cài Môi Trường + Set API Key

In [ ]:
!pip install -e "/content/arag[full]" -q

import os

# ============================================================
# SET YOUR API KEY HERE (or use Colab Secrets)
# ============================================================
# Option A: Direct (not recommended for sharing)
# os.environ["ARAG_API_KEY"] = "sk-..."
# os.environ["ARAG_MODEL"] = "gpt-4o-mini"

# Option B: Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ["ARAG_API_KEY"] = userdata.get("ARAG_API_KEY")
    os.environ["ARAG_MODEL"] = userdata.get("ARAG_MODEL") or "gpt-4o-mini"
    os.environ["ARAG_BASE_URL"] = userdata.get("ARAG_BASE_URL") or "https://api.openai.com/v1"
    print("API key loaded from Colab Secrets")
except Exception:
    print("WARNING: ARAG_API_KEY not set. Set it before running experiments.")

print("API key set:", bool(os.environ.get("ARAG_API_KEY")))
print("Model:", os.environ.get("ARAG_MODEL", "gpt-4o-mini"))
print("Base URL:", os.environ.get("ARAG_BASE_URL", "https://api.openai.com/v1"))

## Cell 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_RESULTS_DIR = "/content/drive/MyDrive/thesis_arag_results"
import os
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {DRIVE_RESULTS_DIR}")

## Cell 4: Build Index HotpotQA / MuSiQue (GPU)

In [ ]:
import os
REPO_DIR = "/content/arag"

# Build MuSiQue index
!python {REPO_DIR}/scripts/build_index.py \
    --chunks {REPO_DIR}/data/musique/chunks.json \
    --output {REPO_DIR}/data/musique/index \
    --model Qwen/Qwen3-Embedding-0.6B \
    --device cuda:0

# Build HotpotQA index
!python {REPO_DIR}/scripts/build_index.py \
    --chunks {REPO_DIR}/data/hotpotqa/chunks.json \
    --output {REPO_DIR}/data/hotpotqa/index \
    --model Qwen/Qwen3-Embedding-0.6B \
    --device cuda:0

print("Indexes built.")

## Cell 5: Chạy Naive RAG

In [ ]:
REPO_DIR = "/content/arag"
LIMIT = 20  # MVP: 20 câu. Đổi thành 100 khi expand
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py \
    --config {REPO_DIR}/configs/thesis/musique_base.yaml \
    --variant naive_rag \
    --questions {REPO_DIR}/data/musique/questions.json \
    --output {REPO_DIR}/results/thesis/naive_rag_musique \
    --limit {LIMIT} --workers {WORKERS}

print("Naive RAG done.")

## Cell 6: Chạy A-RAG Baseline

In [ ]:
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py \
    --config {REPO_DIR}/configs/thesis/musique_base.yaml \
    --variant arag_baseline \
    --questions {REPO_DIR}/data/musique/questions.json \
    --output {REPO_DIR}/results/thesis/arag_baseline_musique \
    --limit {LIMIT} --workers {WORKERS}

print("A-RAG Baseline done.")

## Cell 7: Chạy A-RAG + Entity Tracker

In [ ]:
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py \
    --config {REPO_DIR}/configs/thesis/musique_entity.yaml \
    --variant arag_entity_tracker \
    --questions {REPO_DIR}/data/musique/questions.json \
    --output {REPO_DIR}/results/thesis/arag_entity_tracker_musique \
    --limit {LIMIT} --workers {WORKERS}

print("A-RAG + Entity Tracker done.")

## Cell 8: Chạy A-RAG + Evidence Checker

In [ ]:
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py \
    --config {REPO_DIR}/configs/thesis/musique_evidence.yaml \
    --variant arag_evidence_checker \
    --questions {REPO_DIR}/data/musique/questions.json \
    --output {REPO_DIR}/results/thesis/arag_evidence_checker_musique \
    --limit {LIMIT} --workers {WORKERS}

print("A-RAG + Evidence Checker done.")

## Cell 9: Chạy A-RAG + Full (ET + EV)

In [ ]:
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py \
    --config {REPO_DIR}/configs/thesis/musique_full.yaml \
    --variant arag_entity_evidence_full \
    --questions {REPO_DIR}/data/musique/questions.json \
    --output {REPO_DIR}/results/thesis/arag_entity_evidence_full_musique \
    --limit {LIMIT} --workers {WORKERS}

print("A-RAG Full done.")

## Cell 10: Evaluate và Xuất Bảng So Sánh

In [ ]:
import subprocess, json
REPO_DIR = "/content/arag"

# Quick contain-match comparison (no LLM needed)
!python {REPO_DIR}/scripts/thesis/compare_results.py \
    --results {REPO_DIR}/results/thesis/ \
    --dataset musique

# Optional: LLM-based accuracy evaluation (costs money)
# for variant in ["naive_rag", "arag_baseline", "arag_entity_tracker",
#                 "arag_evidence_checker", "arag_entity_evidence_full"]:
#     !python {REPO_DIR}/scripts/eval.py \
#         --predictions {REPO_DIR}/results/thesis/{variant}_musique/predictions.jsonl \
#         --config {REPO_DIR}/configs/thesis/musique_base.yaml \
#         --workers 5

# Display comparison JSON
import json
cmp_file = f"{REPO_DIR}/results/thesis/comparison_musique.json"
try:
    with open(cmp_file) as f:
        cmp = json.load(f)
    import pandas as pd
    rows = []
    for v, stats in cmp.items():
        if stats:
            rows.append({"variant": v, **stats})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
except Exception as e:
    print(f"Cannot display table: {e}")

## Cell 11: Copy Results về Google Drive

In [ ]:
import shutil, os
REPO_DIR = "/content/arag"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/thesis_arag_results"

src = f"{REPO_DIR}/results/thesis"
dst = f"{DRIVE_RESULTS_DIR}/results_thesis"

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Results copied to: {dst}")
else:
    print("No results to copy yet.")

# List saved files
for root, dirs, files in os.walk(dst):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"  {path.replace(dst, '')} ({size:,} bytes)")